In [1]:
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# PATHS
# =========================
WF_ROOT = Path(r"C:\Dev\Bachelorarbeit\results\accounting\runs\wf_final_2")
OUT_DIR = WF_ROOT / "_plots_wf"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# OPTIONAL: Benchmark, falls du eine separate CSV hast (gleiche Spalten wie metrics_per_test_year.csv)
# Beispiel: BENCH_CSV = Path(r"...\benchmark\...\_report\metrics_per_test_year.csv")
BENCH_CSV = Path(r"C:\Dev\Bachelorarbeit\data\benchmark_fin_per_year")
BENCH_LABEL = "Benchmark"

# =========================
# Helpers
# =========================
def parse_config_label(config_dir: str) -> str:
    """
    Beispiele:
      log_ppo_S0                      -> log-S0
      icvar_ppo_S1_lamda_25            -> icvar-S1 (λ=25)
      icvar_dd_ppo_S0_lamda_15_gamma_1 -> icvar_dd-S0 (λ=15, γ=1)
    """
    s = config_dir.replace("_final", "")
    reward = None
    if s.startswith("log_"):
        reward = "log"
    elif s.startswith("icvar_dd_"):
        reward = "icvar_dd"
    elif s.startswith("icvar_"):
        reward = "icvar"

    m_state = re.search(r"_S([01])", s)
    state = f"S{m_state.group(1)}" if m_state else ""

    m_lam = re.search(r"lamda_(\d+)", s)
    lam = m_lam.group(1) if m_lam else None

    m_gam = re.search(r"gamma_(\d+)", s)
    gam = m_gam.group(1) if m_gam else None

    label = f"{reward}-{state}" if reward else s
    extras = []
    if lam is not None:
        extras.append(f"λ={lam}")
    if gam is not None:
        extras.append(f"γ={gam}")
    if extras:
        label += " (" + ", ".join(extras) + ")"
    return label

def find_latest_reports(root: Path, filename: str):
    """
    Findet alle .../_report/<filename> und nimmt pro config_dir den "latest" run_id
    (falls du mehrere Runs pro Konfi hast).
    """
    files = list(root.rglob(f"_report/{filename}"))
    if not files:
        return []

    # key: config_dir -> best path
    best = {}
    for p in files:
        # rel: <config_dir>/<run_id>/<E2>/ _report / filename
        rel = p.relative_to(root).parts
        if len(rel) < 5:
            continue
        config_dir, run_id = rel[0], rel[1]

        # timestamp am Ende vom run_id (…_YYYYMMDD_HHMMSS)
        m_ts = re.search(r"(\d{8}_\d{6})$", run_id)
        ts = m_ts.group(1) if m_ts else run_id

        key = config_dir
        if key not in best or ts > best[key][0]:
            best[key] = (ts, p)

    return [v[1] for v in best.values()]

# =========================
# 1) Load metrics_per_test_year.csv (alle Konfis)
# =========================
paths_year = find_latest_reports(WF_ROOT, "metrics_per_test_year.csv")
if not paths_year:
    raise FileNotFoundError(f"Keine metrics_per_test_year.csv unter {WF_ROOT} gefunden.")

rows = []
for csv_path in paths_year:
    rel = csv_path.relative_to(WF_ROOT).parts
    config_dir = rel[0]
    run_id = rel[1]

    df = pd.read_csv(csv_path)
    df["config_dir"] = config_dir
    df["config"] = parse_config_label(config_dir)
    df["run_id"] = run_id
    rows.append(df)

df_year = pd.concat(rows, ignore_index=True)

# optional Benchmark hinzufügen
if BENCH_CSV is not None:
    df_b = pd.read_csv(BENCH_CSV)
    df_b["config"] = BENCH_LABEL
    df_b["config_dir"] = "benchmark"
    df_b["run_id"] = "benchmark"
    df_year = pd.concat([df_year, df_b], ignore_index=True)

# sanity
print("Loaded rows:", df_year.shape)
print("Configs:", sorted(df_year["config"].unique()))
print("Years:", sorted(df_year["test_year"].unique()))

# =========================
# 2) Plot: pro Testjahr ein Chart mit allen Konfis + Benchmark
# =========================
metric = "ex_cum_return"
years = sorted(df_year["test_year"].dropna().unique())

# Globale y-limits für bessere Vergleichbarkeit
ymin = df_year[metric].min()
ymax = df_year[metric].max()
pad = 0.05 * (ymax - ymin) if ymax > ymin else 0.1
ylim = (ymin - pad, ymax + pad)

# Reihenfolge der Konfis (optional)
# wenn du fixe Reihenfolge willst: bau dir hier eine Liste und sortiere danach.
# Default: alphabetisch, Benchmark ans Ende.
def sort_key(cfg):
    return (cfg == BENCH_LABEL, cfg)

for y in years:
    sub = df_year[df_year["test_year"] == y].copy()
    sub = sub.dropna(subset=[metric])

    # 1 Wert pro Konfi pro Jahr erwartet
    sub = sub.groupby("config", as_index=False)[metric].mean()
    sub = sub.sort_values("config", key=lambda s: s.map(sort_key))

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.bar(sub["config"], sub[metric])
    ax.axhline(0, linewidth=1)

    ax.set_title(f"Walk-Forward: Excess Cumulative Return – Testjahr {y}")
    ax.set_ylabel("ex_cum_return")
    ax.set_ylim(*ylim)
    ax.grid(True, axis="y", alpha=0.2)

    ax.tick_params(axis="x", labelrotation=20)
    fig.tight_layout()

    out_png = OUT_DIR / f"wf_ex_cum_return_{y}.png"
    out_pdf = OUT_DIR / f"wf_ex_cum_return_{y}.pdf"
    fig.savefig(out_png, dpi=200)
    fig.savefig(out_pdf)
    plt.close(fig)

print("Saved year charts ->", OUT_DIR)

# =========================
# 3) KPI-Tabelle: Konfi x KPI-Spalten
#    Prefer: metrics_per_path.csv (stitch über alle WF-Jahre)
#    Fallback: Mittelwert über Testjahre aus metrics_per_test_year.csv
# =========================
paths_overall = find_latest_reports(WF_ROOT, "metrics_per_path.csv")

if paths_overall:
    rows = []
    for csv_path in paths_overall:
        config_dir = csv_path.relative_to(WF_ROOT).parts[0]
        run_id = csv_path.relative_to(WF_ROOT).parts[1]
        df = pd.read_csv(csv_path)

        # falls mehrere runs in file: nimm alle und markiere config
        df["config"] = parse_config_label(config_dir)
        df["config_dir"] = config_dir
        df["run_id"] = run_id
        rows.append(df)

    df_kpi = pd.concat(rows, ignore_index=True)

    # wenn noch "run" (E2) drin ist und du nur E2 willst:
    if "run" in df_kpi.columns:
        df_kpi = df_kpi[df_kpi["run"].isin(["E2", "SB3_Defaults"])]

    # KPI-Spalten: alles numerische außer IDs
    id_cols = {"config","config_dir","run_id","run","path","fold","test_year","test_dir"}
    kpi_cols = [c for c in df_kpi.columns if c not in id_cols]

    # 1 Zeile pro config (falls mehrere Zeilen: Mittelwert)
    df_kpi = df_kpi.groupby("config", as_index=True)[kpi_cols].mean()

else:
    # Fallback: KPI-Tabelle als Durchschnitt über Jahre (nicht perfekt für Sharpe, aber als Notlösung ok)
    id_cols = {"config","config_dir","run_id","run","fold","test_year","test_dir"}
    kpi_cols = [c for c in df_year.columns if c not in id_cols]

    df_kpi = df_year.groupby("config", as_index=True)[kpi_cols].mean()

# Benchmark optional hinzufügen (falls vorhanden und nicht schon drin)
if BENCH_LABEL in df_year["config"].unique() and BENCH_LABEL not in df_kpi.index:
    df_bk = df_year[df_year["config"] == BENCH_LABEL].groupby("config")[df_kpi.columns].mean()
    df_kpi = pd.concat([df_kpi, df_bk], axis=0)

# speichern
out_csv = OUT_DIR / "wf_kpis_table.csv"
df_kpi.to_csv(out_csv)

# optional latex
out_tex = OUT_DIR / "wf_kpis_table.tex"
df_kpi.to_latex(out_tex, float_format="%.4f")

print("Saved KPI table ->", out_csv)
print("Saved LaTeX table ->", out_tex)


PermissionError: [Errno 13] Permission denied: 'C:\\Dev\\Bachelorarbeit\\data\\benchmark_fin_per_year'